# How to make quantum cheese with Construct
### From a compact geometry model to a reversible circuit

How can a short quantum circuit describe a material containing a large number of pores? In this tutorial, we build a reproducible porous geometry and turn it into a circuit that answers **“Is this grid site inside a hole?”**

Along the way, you will learn to use **PsiQuantum Construct**: write circuits in Python with **PsiQDK Workbench**, group them into reusable circuit routines called **Qubricks**, inspect them with **Circuit Designer**, and investigate their cost with **Resource Analyzer**.

The construction is based on [*How to make quantum cheese: efficient geometry oracles for exponentially many pseudorandom microstructures*, arXiv:2606.00222](https://arxiv.org/abs/2606.00222). We develop the definitions and reasoning here; you do not need to read the paper first.

**Prerequisites:** Python, binary integers, computational-basis states, and the idea of a controlled gate. No familiarity with cryptography, Construct, or quantum PDE algorithms is assumed.

**What you will have built:** an oracle with the action
$$
O_g:|x\rangle|b\rangle|0\rangle_{\mathrm{work}}
\longmapsto |x\rangle|b\oplus g(x)\rangle|0\rangle_{\mathrm{work}},
$$
where $g(x)=1$ means **void**, $g(x)=0$ means **material**, and $\oplus$ is XOR. The coordinate is preserved and temporary ancillary qubits are returned to zero. Ancillary qubits are temporary registers used during the calculation; they are not part of its input or output.


## The journey

| Step | Geometry idea | Construct skill |
| --- | --- | --- |
| 1. Define the material | Select centres and place a stencil around them | Establish an independent classical reference |
| 2. Query one coordinate | Reversible Boolean evaluation | Create a QPU and binary registers |
| 3. Mix the addresses | A keyed, invertible permutation | Compose arithmetic and custom Qubricks |
| 4. Carve the holes | Count contributions from shifted centres | Accumulate, threshold, and uncompute |
| 5. Inspect and validate | Check the actual oracle against its definition | Draw circuits and capture/play back instructions |
| 6. Understand the cost | Vary address width, rounds, and stencil size | Compare numeric estimates and explore call graphs |
| 7. Experiment | Change shapes and weighted local rules | Reuse the same circuit components |

The first part runs with ordinary scientific Python. Construct imports appear when we begin building circuits. Optional browser viewers and longer circuit scans are disabled by default, so **Run All** follows a compact path.


## 0. Set up your environment

For the whole notebook, use a supported PsiQDK installation. Follow the current [installation instructions](https://docs.construct.psiquantum.com/psiqdk/installation.html); the documented installation command is `python -m pip install psiqdk`. Install `numpy pandas matplotlib ipykernel` in the same environment and select that environment as the notebook kernel. On Windows, use WSL with a supported Linux distribution.

| Part of Construct | What we use it for |
| --- | --- |
| PsiQDK Workbench | Registers, gates, reusable routines, simulation, and resource estimates |
| PsiQDK Algorithms | A reusable comparison routine |
| Circuit Designer | Explore the hierarchy of a circuit visually |
| Resource Analyzer | Explore the hierarchy of its resource costs |
| PsiQDK Visualize | Embed supported interactive viewers inside a notebook |

We use the Python interface throughout. The optional exports also let you open the same program in Construct's browser tools.



In [ ]:
import json
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Turn these on when you want the optional parts of the workflow.
RUN_INTERACTIVE_VIEWERS = False
EXPORT_CONSTRUCT_FILES = False
RUN_FULL_CIRCUIT_IMAGE = False
RUN_RESOURCE_SWEEPS = False
SAVE_PAPER_FIGURES = False
export_dir = Path("quantum_cheese_exports")
export_dir.mkdir(exist_ok=True)
if SAVE_PAPER_FIGURES:
    fig_dir = "figures"
    (export_dir / fig_dir).mkdir(exist_ok=True)


## 1. Why does geometry need a quantum circuit?

Imagine a diffusion problem on a porous material. Discretization replaces a continuous domain by grid sites. The equations at each site depend on whether that site contains material. A quantum algorithm for these partial differential equations (PDEs) therefore needs to query the geometry at a binary grid address.

A classical array can store one Boolean per site. But with $N=2^n$ sites, that array has $N$ entries while an address needs only $n$ bits. An arbitrary array can therefore be exponentially large in the address width.

The paper considers increasing the material volume at a fixed microscopic resolution. More address bits then describe more sites and, at fixed centre density, more pores. Listing every pore would require an exponentially long description.

Here we use a **procedural geometry**: a short rule generates the material. We can evaluate that rule classically or compile it into a reversible circuit. A reversible circuit retains enough information to undo its operations, which lets us restore the input and its ancillary qubits after computing the answer.

For a superposition of addresses, the oracle acts linearly:
$$
\sum_x a_x|x\rangle|0\rangle
\longmapsto\sum_x a_x|x\rangle|g(x)\rangle.
$$
The $a_x$ are quantum amplitudes. This is **coherent access**: the query works without measuring the address. Reading the entire geometry still requires many queries or measurements. We build the geometry component here; a complete PDE algorithm also needs its discretized operator, solver, and output procedure.


### 1.1 Notation

We use a periodic two-dimensional grid: moving past one edge wraps to the opposite edge. Later circuit routines also accept more dimensions.

| Symbol | Meaning | Notebook name |
| --- | --- | --- |
| $n_x,n_y$ | Bits per coordinate | `nx`, `ny` |
| $n=n_x+n_y$ | Total address width, required to be even here | `nx + ny` |
| $N=2^n$ | Number of grid sites | `2**(nx + ny)` |
| $w=n/2$ | Width of each half of the address permutation | `half_width` |
| $m$ | Number of permutation rounds, or repeated mixing steps | `len(keys)` |
| $M$ | Number of selected hole centres | `M` |
| $\Delta$ | Hole stencil: the offsets removed around one centre | `deltas` |
| $R=|\Delta|$ | Number of stencil offsets | `len(deltas)` |
| $S$ | Accumulator width, introduced when we count overlapping holes | `S` |

Coordinates satisfy $0\le x<2^{n_x}$ and $0\le y<2^{n_y}$. We pack them into one integer address:
$$
a(x,y)=x+2^{n_x}y.
$$
The $x$ bits are the low bits. Spatial shifts wrap independently modulo $2^{n_x}$ and $2^{n_y}$. This geometry convention does not choose all boundary conditions for a later PDE.


### 1.2 Select centres, count contributions, then threshold

A **permutation** rearranges the $N$ addresses without repetition. We choose a permutation $P_k$ specified by a classical key sequence $k$. It is **pseudorandom**: the sequence determines the result exactly, but the resulting spatial pattern is intended to look disordered. Changing the keys changes the material realization.

Define the **centre predicate**, a Boolean function that tests whether a site is selected:
$$
c_{k,M}(x,y)=\mathbf1[P_k(a(x,y))<M].
$$
Here $\mathbf1[\cdot]$ is 1 when its condition is true and 0 otherwise. Exactly $M$ permutation outputs are below $M$, so there are exactly $M$ centres. This does not imply uniform sampling over all possible sets of $M$ sites.

A centre at $z$ removes every site $z+\delta$ with $\delta\in\Delta$. To test a site $x$, work backwards: its possible centres are $x-\delta$. Here and below, $x$ can denote the full coordinate pair. Count how many of these centres are present:
$$
s(x)=\sum_{\delta\in\Delta}c_{k,M}(x-\delta),\qquad
g(x)=\mathbf1[s(x)\ge1].
$$
This count is the **accumulator**. Thresholding at one implements the Boolean OR: a site is void if at least one hole covers it. In the circuit, each centre test produces a one-qubit **flag**, used immediately to increment the accumulator and then erased by reversing the test. We do not store one flag per offset.

Overlaps increase the count but remove a site only once. Thus $M$ counts centres, not connected pores. The centre fraction $M/N$ generally differs from the void fraction $N^{-1}\sum_xg(x)$. We plot $1-g(x)$ when black should indicate void.


### 1.3 A classical reference for the keyed permutation

We use the round function of **SPECK**, a cipher that transforms a message using a key. Here its role is to mix grid addresses reversibly. We supply one classical key per round; no separate key-generation procedure or cryptographic security claim is needed.

Split the address into two $w$-bit integers, $L$ and $R$. A bit rotation moves each bit a fixed number of places, wrapping at the end of the word. Write $\operatorname{ROR}_\alpha$ for rotation right by $\alpha$ and $\operatorname{ROL}_\beta$ for rotation left by $\beta$. One round is
$$
L'=\big((\operatorname{ROR}_\alpha(L)+R)\bmod2^w\big)\oplus k_j,
\qquad
R'=\operatorname{ROL}_\beta(R)\oplus L'.
$$
The addition wraps modulo $2^w$, and XOR acts bit by bit. The rotation distances $\alpha,\beta$ are called $u,v$ in the paper.

The low half of the address is $L$ and the high half is $R$. For unequal coordinate widths, these halves need not coincide with coordinate registers. Spatial shifts still act on the original axes.

The next functions give an independent classical reference. We will build the reversible version in Section 3.


In [ ]:
# Independent classical reference for the geometry oracle.

def rotate_left(value, distance, width):
    """Rotate a width-bit unsigned integer left; bit zero is least significant."""
    distance %= width
    mask = (1 << width) - 1
    return ((value << distance) | (value >> ((width - distance) % width))) & mask


def speck_reference(address, total_bits, keys, alpha=1, beta=1):
    """Evaluate the keyed permutation on a single integer address."""
    if total_bits < 2 or total_bits % 2:
        raise ValueError("The total address width must be positive and even")
    width = total_bits // 2
    mask = (1 << width) - 1
    left, right = address & mask, address >> width
    for key in keys:
        if not 0 <= int(key) <= mask:
            raise ValueError("Each key must fit in one permutation half")
        left = ((rotate_left(left, -alpha, width) + right) & mask) ^ int(key)
        right = rotate_left(right, beta, width) ^ left
    return left | (right << width)


def geometry_reference(nx, ny, keys, M, deltas, rot=(1, 1)):
    """Return centre and void arrays; rows are y, columns are x."""
    if nx < 1 or ny < 1 or (nx + ny) % 2:
        raise ValueError("Use positive axis widths and an even total width")
    if not 0 <= M <= 2**(nx + ny):
        raise ValueError("M must be between zero and the number of sites")
    centres = np.zeros((2**ny, 2**nx), dtype=np.uint8)
    for y in range(2**ny):
        for x in range(2**nx):
            address = x + (y << nx)
            centres[y, x] = speck_reference(address, nx + ny, keys, *rot) < M
    score = np.zeros_like(centres, dtype=np.int64)
    for dx, dy in deltas:
        # np.roll evaluates centres at (x - dx, y - dy), with periodic wrapping.
        score += np.roll(centres, shift=(dy, dx), axis=(0, 1))
    return centres, (score >= 1).astype(np.uint8)


### First look: a material described by a handful of numbers

These reproduce the parameter choices of the paper: a 64 × 64 grid, 512 centres, five rounds, rotations (1, 7), and the four-offset stencil shown below (Figure 2 of the manuscript). The preview below is evaluated **classically**, so you can see the target without waiting for thousands of circuit queries. Later we will compare the actual circuit to the same reference on a smaller grid.

The stencil we use is asymmetric: each centre removes itself, its left neighbour, and its two vertical neighbours. Asymmetry in the example below helps us to check for correctness of sign and axis convention.


In [ ]:
preview_nx, preview_ny = 6, 6
preview_keys = (38, 51, 28, 14, 42)
preview_M = 512
preview_deltas = ((0, 0), (-1, 0), (0, 1), (0, -1))
preview_rot = (1, 7)
preview_centres, preview_void = geometry_reference(preview_nx, preview_ny, preview_keys, preview_M, preview_deltas, preview_rot)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(preview_centres, cmap="Blues", origin="lower", interpolation="nearest", vmin=0, vmax=1)
axes[1].imshow(1 - preview_void, cmap="gray", origin="lower", interpolation="nearest", vmin=0, vmax=1)
axes[0].set(title=f"{preview_centres.sum()} centre sites", xlabel="x", ylabel="y")
axes[1].set(title=f"Porous material: {preview_void.mean():.1%} void", xlabel="x", ylabel="y")
plt.tight_layout()
plt.show()


**Read the stencil as a shape.** Each black square is an offset removed around a centre; the red outline identifies offset (0, 0). A centre at (a, b) translates this entire shape to (a, b). The circuit works backwards from a queried pixel to those possible centres.


In [ ]:
stencil_image = np.zeros((3, 3), dtype=np.uint8)
for dx, dy in preview_deltas:
    stencil_image[dy + 1, dx + 1] = 1
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.imshow(stencil_image, cmap="gray_r", origin="lower", extent=(-1.5, 1.5, -1.5, 1.5), vmin=0, vmax=1)
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((-0.5, -0.5), 1, 1, fill=False, edgecolor="tab:red", linewidth=2))
ax.set(title="Offsets removed by one centre", xlabel="Offset in x", ylabel="Offset in y", xticks=(-1, 0, 1), yticks=(-1, 0, 1))
fig.tight_layout()
if SAVE_PAPER_FIGURES:
    namefile = "stencil_plot.pdf"
    fig.savefig(export_dir / fig_dir / namefile)
fig.show()


### 1.4 Work through a tiny grid

For the running example, use a 4 × 4 grid, three centres, and a two-pixel horizontal hole $\Delta=\{(0,0),(1,0)\}$. The arithmetic is small enough to inspect every address.

**Before running:** the centre count must be three. Can the number of void sites exceed six? Can it be smaller than six? Each centre contributes at most two sites, and overlaps can reduce the total.

Coordinate shifts wrap modulo 4 on each axis; the packed address ranges from 0 to 15.


In [ ]:
toy_nx = toy_ny = 2
toy_keys = (1, 2)
toy_M = 3
toy_deltas = ((0, 0), (1, 0))
toy_rot = (1, 1)
toy_centres, toy_void = geometry_reference(toy_nx, toy_ny, toy_keys, toy_M, toy_deltas, toy_rot)

address_grid = np.arange(16).reshape(4, 4)
permuted_grid = np.array([speck_reference(a, 4, toy_keys, *toy_rot) for a in range(16)]).reshape(4, 4)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
panels = [(address_grid, "1. Address a = x + 4y"), (permuted_grid, "2. Permutation P(a)"), (toy_centres, "3. Centres: P(a) < 3"), (toy_void, "4. Void after carving")]
for ax, (array, title) in zip(axes, panels):
    ax.imshow(array, cmap="Blues", origin="lower", interpolation="nearest")
    for y in range(4):
        for x in range(4):
            ax.text(x, y, str(array[y, x]), ha="center", va="center", color="white" if array[y, x] > array.max() / 2 else "black")
    ax.set(title=title, xlabel="x", ylabel="y", xticks=range(4), yticks=range(4))
fig.tight_layout()
if SAVE_PAPER_FIGURES:
    namefile = "shuffle_carve.pdf"
    fig.savefig(export_dir / fig_dir / namefile)
fig.show()
assert toy_centres.sum() == toy_M
assert toy_M <= toy_void.sum() <= min(16, toy_M * len(toy_deltas))


**Optional paper figure.** The next cell exports the three centre-selection panels separately when `SAVE_PAPER_FIGURES` is enabled. It uses the same data as the preceding example.


In [ ]:
if SAVE_PAPER_FIGURES:
    fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
    panels = [(address_grid, "1. Address a = x + 4y"), (permuted_grid, "2. Permutation P(a)"), (toy_centres, "3. Centres: P(a) < 3")]
    for ax, (array, title) in zip(axes, panels):
        ax.imshow(array, cmap="Blues", origin="lower", interpolation="nearest")
        for y in range(4):
            for x in range(4):
                ax.text(x, y, str(array[y, x]), ha="center", va="center", color="white" if array[y, x] > array.max() / 2 else "black")
        ax.set(title=title, xlabel="x", ylabel="y", xticks=range(4), yticks=range(4))
    fig.tight_layout()
    if SAVE_PAPER_FIGURES:
        namefile = "shuffle.pdf"
        fig.savefig(export_dir / fig_dir / namefile)
    fig.show()


**Trace one query.** We choose a centre and query the site immediately to its right, so the offset $(1,0)$ must detect that centre. If the chosen centre lies on the last column, the query wraps to the first column. The table lists every possible centre relevant to this one pixel.


In [ ]:
centre_y, centre_x = np.argwhere(toy_centres)[0]
query_x, query_y = (int(centre_x) + 1) % 4, int(centre_y)
trace_rows = []
for dx, dy in toy_deltas:
    sx, sy = (query_x - dx) % 4, (query_y - dy) % 4
    address = sx + 4 * sy
    permuted = speck_reference(address, 4, toy_keys, *toy_rot)
    trace_rows.append({"offset": (dx, dy), "possible centre": (sx, sy), "address": address, "P(address)": permuted, "P < M": int(permuted < toy_M)})
display(pd.DataFrame(trace_rows))
score = sum(row["P < M"] for row in trace_rows)
print(f"Query ({query_x}, {query_y}): accumulator = {score}; void = {int(score >= 1)}")


**Checkpoint.** The keys choose a realization, $M$ fixes the number of centres, and the stencil fixes the hole shape. Each query counts covering centres and tests whether the count is at least one.

We evaluate only the possible centres relevant to the queried site. No stored centre list is needed. Next we turn this calculation into a circuit that preserves its input and restores its ancillary qubits.


## 2. Your first Workbench program

A `QPU` is the program's execution and instruction-processing context. Creating one here does not submit a job to quantum hardware. A `Qubits` object is a register attached to that context; its slices and unions refer to existing qubits.

A **filter** processes instructions as Workbench emits them. Our execution pipeline chains two filters:

- `>>buffer>>` records instructions for inspection and circuit drawing.
- `>>bit-sim>>` evaluates computational-basis inputs efficiently.

Later we add `>>witness>>`, a filter that records resource costs. The bit simulator is appropriate because our tests use Boolean arithmetic on basis inputs. It does not simulate a superposition; the unitary action of the circuit specifies what happens on superpositions.

`CuccaroAdd`, `GidneyAdd`, `NaiveAdd`, and `ConditionallyCleanAdd` are alternative modular adders. We use Cuccaro first and compare them later. `CondCleanLessThanConst` compares a register with a classical constant. Its conditionally clean method temporarily uses qubits known to be zero within particular branches of the calculation, then restores them.

[More about execution pipelines](https://docs.construct.psiquantum.com/psiqdk-workbench/new-tutorials/Configuring-Execution.html).


In [ ]:
from psiqdk.workbench import QPU, Qubits, Qubrick
from psiqdk.workbench.arithmetic import CuccaroAdd, GidneyAdd, NaiveAdd, ConditionallyCleanAdd
from psiqdk.algorithms import CondCleanLessThanConst
from psiqdk.workbench.qre import resource_estimator

for package in ("psiqdk", "numpy", "pandas", "matplotlib"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: no distribution metadata found")


### 2.1 Binary registers and controlled XOR

Workbench uses least-significant-bit indexing for integer registers: bit 0 has weight 1, bit 1 has weight 2, and so on. A mask `0b101` therefore addresses bits 0 and 2.

The following tiny program writes the test input 3, applies a bit mask, and copies the resulting register into another clean register using pairwise controlled X gates (CNOTs). An X gate flips a bit; a controlled X does so only when its control is 1. Predict $3\oplus5$ before running it.

`write()` and `read()` are preparation and inspection tools for these basis-state tests. They belong outside the oracle: the coherent circuit itself will contain no reads or input resets.


In [ ]:
first_qpu = QPU(num_qubits=8, filters=[">>buffer>>", ">>bit-sim>>"])
a = Qubits(3, "a", first_qpu)
b = Qubits(3, "b", first_qpu)
a.write(3)
a.x(tgt_bits=0b101)
b.x(cond_zip=a)
print(f"a = {a.read()}, b = {b.read()}; expected 6, 6")
assert (a.read(), b.read()) == (6, 6)
first_qpu.draw()


### 2.2 A Qubrick is a reusable circuit routine

A `Qubrick` groups instructions under a meaningful name. Define its forward action in `_compute`, invoke it with `compute`, and use `uncompute` when you need Workbench to reverse the recorded computation. Its mathematical notation is the adjoint, $U^\dagger$, of the forward operation $U$. This hierarchy will later appear in both circuit diagrams and cost breakdowns.

Our first Qubrick applies a **classical** mask fixed when the program is constructed. There is no quantum key register. XOR is its own inverse, so applying this key twice restores the original value.


In [ ]:
class ApplyKey(Qubrick):
    """XOR a classical integer mask into a quantum register."""

    def _compute(self, target_register: Qubits, key: int):
        if not 0 <= int(key) < (1 << target_register.num_qubits):
            raise ValueError("The key must fit in the target register")
        target_register.x(tgt_bits=int(key))


key_qpu = QPU(num_qubits=5, filters=[">>buffer>>", ">>bit-sim>>"])
key_reg = Qubits(3, "word", key_qpu)
key_reg.write(3)
key_step = ApplyKey(name="key_mask")
key_step.compute(key_reg, 5)
assert key_reg.read() == 6
key_step.uncompute()
assert key_reg.read() == 3
print("Forward XOR gives 6; uncompute restores 3.")


### 2.3 Why compute–copy–uncompute?

Suppose a reversible subroutine $U$ computes a flag while modifying a work register. Merely using that flag and leaving the work register behind can retain information about the input. On a superposition, that information can remain entangled with the coordinate: the joint state cannot be separated into independent coordinate and ancillary states. This can disrupt subsequent operations.

Instead, compute the flag, XOR it into the desired output, then apply $U^\dagger$:
$$
\begin{aligned}
|x\rangle|0\rangle_W|b\rangle
&\xrightarrow{U}|\phi_x\rangle_{XW}|b\rangle\\
&\xrightarrow{\mathrm{controlled}\ X}|\phi_x\rangle_{XW}|b\oplus g(x)\rangle\\
&\xrightarrow{U^\dagger}|x\rangle|0\rangle_W|b\oplus g(x)\rangle.
\end{aligned}
$$
In the middle lines, $|\phi_x\rangle_{XW}$ abbreviates the transformed input together with all temporary registers. Reversing the computation restores both.

“Copy” here means controlled XOR of a basis-valued predicate. Applied coherently, this can entangle the output with the input; it does not clone an arbitrary quantum state.

For each offset, we compute a centre flag, use it to increment the accumulator, then uncompute that flag. After thresholding the total, we reverse the entire accumulation. The middle operation can therefore be a controlled addition rather than a copy.


## 3. Build the reversible address permutation

The reason for using a permutation is now visible: a threshold selects exactly $M$ centres, without collisions or a stored centre list. A SPECK-like construction is convenient because its ingredients have compact reversible implementations.

Every round is invertible:

| Forward operation | Inverse operation |
| --- | --- |
| Rotate bits | Rotate in the opposite direction |
| Add modulo $2^w$, retaining the addend | Subtract modulo $2^w$ |
| XOR a classical key | XOR the same key |
| XOR one register into another, retaining the control | Repeat the same controlled XOR |

Their composition is therefore a permutation. More rounds may improve mixing and increase cost; invertibility alone does not establish cryptographic security or good spatial statistics.


### 3.1 Follow one round in integers

Take $w=3$, $L=5$ (`101`), $R=3$ (`011`), and $k=6$ (`110`). With both rotation distances equal to one:

1. Rotate $L$ right: `101` becomes `110`, or 6.
2. Add $R$ modulo 8: $6+3\equiv1$.
3. XOR the key: $1\oplus6=7$.
4. Rotate $R$ left: `011` becomes `110`, or 6.
5. XOR the updated $L$ into $R$: $6\oplus7=1$.

The round maps $(5,3)$ to $(7,1)$. We will test the circuit against this result.


### 3.2 Rotations and modular addition

The helper below rotates bit **values** using SWAP gates. It is a cyclic permutation of the register bits, not a spatial shift of the grid. The spatial shifts later use modular addition instead.

Our sign convention is the integer convention: positive `distance` rotates left. 

A SWAP exchanges two qubit values. The paper treats bit rotations as free wire re-indexing; this helper emits explicit SWAP instructions. Their treatment in compiled resource estimates depends on the pipeline, so that convention must be checked before comparing exact counts.

For modular addition, `CuccaroAdd().compute(L, R)` updates the **first** register: $L\leftarrow L+R\pmod{2^w}$, preserving $R$. The routine manages its arithmetic ancillary qubits. We will compare several available adder implementations after the full oracle is working.


In [ ]:
def rotate_register_left(qreg, distance):
    """Rotate bit values left; negative distance rotates right."""
    reverse_swaps = distance > 0
    distance = (qreg.num_qubits - (abs(distance) % qreg.num_qubits)) % qreg.num_qubits
    if distance == 0:
        return
    permutation = [(bit + distance) % qreg.num_qubits for bit in range(qreg.num_qubits)]
    swaps = []
    for index in range(qreg.num_qubits):
        partner = permutation[index:].index(index) + index
        if partner != index:
            swaps.append((index, partner))
            permutation[partner] = permutation[index]
    if reverse_swaps:
        swaps.reverse()
    for first, second in swaps:
        qreg[first].swap(qreg[second])


class Speck(Qubrick):
    """Compose keyed reversible mixing rounds on equal-width words."""

    def __init__(self, adderclass=CuccaroAdd, **kwargs):
        self.adderclass = adderclass
        super().__init__(**kwargs)

    def _compute(self, L: Qubits, R: Qubits, keys, alpha=1, beta=1):
        if L.num_qubits != R.num_qubits:
            raise ValueError("Permutation halves must have equal widths")
        for round_id, key in enumerate(keys):
            rotate_register_left(L, -alpha)
            self.adderclass(name=f"add_{round_id}").compute(L, R)
            ApplyKey(name=f"key_{round_id}").compute(L, int(key))
            rotate_register_left(R, beta)
            R.x(cond_zip=L)


In [ ]:
round_qpu = QPU(num_qubits=12, filters=[">>buffer>>", ">>bit-sim>>"])
left = Qubits(3, "L", round_qpu)
right = Qubits(3, "R", round_qpu)
left.write(5)
right.write(3)
one_round = Speck(name="one_round")
one_round.compute(left, right, keys=(6,), alpha=1, beta=1)
assert (left.read(), right.read()) == (7, 1)
print("One round: (5, 3) ->", (left.read(), right.read()))
one_round.uncompute()
assert (left.read(), right.read()) == (5, 3)
print("Uncompute: restored (5, 3).")


### 3.3 Inspect the same program at two levels

A small, zero-initialized example makes the hierarchy easy to inspect without including test-input preparation in the picture. `draw(show_qubricks=True)` displays named routine boxes. Identify the additions and key applications inside the permutation before expanding the elementary gates.

This is a useful Construct workflow: start at the level of the algorithm, then inspect the blocks that implement it.


In [ ]:
permutation_qpu = QPU(num_qubits=8, filters=[">>buffer>>", ">>bit-sim>>"])
perm_left = Qubits(3, "L", permutation_qpu)
perm_right = Qubits(3, "R", permutation_qpu)
Speck(name="keyed_permutation").compute(perm_left, perm_right, keys=(6, 3), alpha=1, beta=1)
permutation_qpu.draw(show_qubricks=True)


**Optional: Circuit Designer.** Enable `RUN_INTERACTIVE_VIEWERS` in the setup cell to embed the hierarchical viewer. Expand the permutation and locate a round's adder. Enable `EXPORT_CONSTRUCT_FILES` to write a `.circuit` file that can be opened in [Circuit Designer](https://construct.psiquantum.com/cd).

The browser tool and the notebook view are two ways to inspect the same Workbench program. Opening a viewer is not needed to simulate or estimate it.


In [ ]:
if RUN_INTERACTIVE_VIEWERS or EXPORT_CONSTRUCT_FILES:
    from psiqdk.workbench.integrations import circuit_designer
if RUN_INTERACTIVE_VIEWERS:
    display(circuit_designer.draw(permutation_qpu, expanded="qubricks"))
if EXPORT_CONSTRUCT_FILES:
    with (export_dir / "keyed_permutation.circuit").open("w") as stream:
        json.dump(circuit_designer.export(permutation_qpu, depth=4), stream)
    print("Exported", export_dir / "keyed_permutation.circuit")


## 4. From one centre query to the complete hole oracle

We compose three routines, matching the paper's shifted query $Q_r$, accumulation $Z$, and geometry oracle $G$:

| Routine | Computation | What remains after its forward call |
| --- | --- | --- |
| `ShiftCentreCompute` | Test $c_{k,M}(x-\delta)$ | A flag and a transformed coordinate; call `uncompute()` to restore both |
| `HoleCompute` | Add one for every covering centre | Original coordinates and the count in an accumulator |
| `HoleOracle` | XOR $\mathbf1[s\ge1]$ into the output | Original coordinates and updated output; all internal ancillary qubits restored |

Coordinate registers are supplied as a list, so the routines also support more than two dimensions. The SPECK permutation requires an even total address width. The packed address is a view of the coordinate qubits, not a copy.


### 4.1 Query one possible centre

For one offset $\delta$:

1. Subtract each component from its coordinate register modulo that axis length.
2. Join the coordinate registers into an address **view**, without copying qubits.
3. Permute that address with a key $k$ (common for all $\delta$).
4. Compare its value to $M$, producing a flag.

The coordinate has been modified during the forward computation. The caller uses the flag to control an accumulator increment, then calls `uncompute()` to reverse the comparison, permutation, and shifts.

In Workbench, `set_result_qreg` identifies the routine's result register; it does not measure it. `CondCleanLessThanConst` supplies the comparison against a classical constant through its `targ` register.

The endpoints $M=0$ and $M=N$ are constant predicates, so we handle them directly rather than asking the comparator to represent an out-of-range constant.


In [ ]:
class ShiftCentreCompute(Qubrick):
    """Compute c_k(x - delta); use uncompute to restore all intermediates."""

    def __init__(self, comparatorClass=CondCleanLessThanConst, adderclass=CuccaroAdd, **kwargs):
        self.comparatorClass = comparatorClass
        self.adderclass = adderclass
        super().__init__(**kwargs)

    def _compute(self, Xs, keys, delta, rot, M):
        Xs = list(Xs)
        if not Xs or len(Xs) != len(delta):
            raise ValueError("Supply one offset component per coordinate register")
        total_bits = sum(X.num_qubits for X in Xs)
        if total_bits % 2:
            raise ValueError("The total coordinate width must be even")
        if not 0 <= M <= (1 << total_bits):
            raise ValueError("M must be between zero and the number of sites")
        if M in (0, 1 << total_bits):
            self.flag = self.alloc_temp_qreg(1, "constant_flag")
            if M:
                self.flag.x()
            self.set_result_qreg(self.flag)
            return
        for axis, (X, offset) in enumerate(zip(Xs, delta)):
            shift = (-int(offset)) % (1 << X.num_qubits)
            self.adderclass(name=f"shift_{axis}").compute(X, shift)
        full = Xs[0]
        for X in Xs[1:]:
            full = full | X
        half_width = total_bits // 2
        self.permutation = Speck(name="permutation", adderclass=self.adderclass)
        self.permutation.compute(full[:half_width], full[half_width:], keys, alpha=rot[0], beta=rot[1])
        self.comparison = self.comparatorClass()
        self.comparison.compute(full, M)
        self.flag = self.comparison.targ
        self.set_result_qreg(self.flag)


### 4.2 Count covering centres

For $R$ stencil offsets, the count is
$$
s(x)=\sum_{r=1}^R c_{k,M}(x-\delta_r),\qquad0\le s(x)\le R.
$$
For each offset, compute its flag, add 1 to the accumulator if the flag is 1, then uncompute the centre query. This restores the coordinate before testing the next offset and allows the flag's ancillary qubits to be reused.

The paper allows signed kernel weights, so it uses **two's-complement encoding**: an $S$-bit register represents integers from $-2^{S-1}$ to $2^{S-1}-1$. We use the same convention here, even though these unit-weight counts are nonnegative. Choosing $2^{S-1}>R$, implemented as `S = R.bit_length() + 1`, prevents overflow. The width grows logarithmically with the number of offsets.

`HoleCompute` leaves the accumulator as its result. It does not yet produce the geometry bit.


In [ ]:
class HoleCompute(Qubrick):
    """Count covering centres, restoring coordinates after each contribution."""

    def __init__(self, comparatorClass=CondCleanLessThanConst, adderclass=CuccaroAdd, **kwargs):
        self.comparatorClass = comparatorClass
        self.adderclass = adderclass
        super().__init__(**kwargs)

    def _compute(self, Xs, keys, deltas, rot, M):

        Xs = list(Xs)
        deltas = tuple(tuple(delta) for delta in deltas)
        keys = tuple(keys)
        if not deltas:
            raise ValueError("Use a nonempty stencil")
        S = len(deltas).bit_length() + 1
        self.accumulator = self.alloc_temp_qreg(S, "accumulator")

        for offset_id, delta in enumerate(deltas):
            check = ShiftCentreCompute(name=f"offset_{offset_id}", comparatorClass=self.comparatorClass, adderclass=self.adderclass)
            check.compute(Xs, keys, delta, rot, M)
            self.adderclass(name=f"accum_{offset_id}").compute(self.accumulator, 1, ctrl=check.flag)
            check.uncompute()

        self.set_result_qreg(self.accumulator)


### 4.3 Threshold the count and clean the accumulator

A site is void when $s\ge1$. The comparator returns $[s<1]$, so we XOR that flag into the output and apply X to complement it:
$$
b\oplus[s<1]\oplus1=b\oplus[s\ge1].
$$
All attainable counts here are nonnegative, so an unsigned comparison is sufficient. Signed weighted sums require the conversion described in Section 8.

Uncompute the comparison **while the count is still present**, then uncompute `HoleCompute`. This last step reverses all contributions and returns the accumulator to zero. Only the external output remains changed. These reverse operations are part of the oracle's cost.


In [ ]:
class HoleOracle(Qubrick):
    """XOR the thresholded count into the output and restore ancillary qubits."""

    def __init__(self, comparatorClass=CondCleanLessThanConst, adderclass=CuccaroAdd, **kwargs):
        self.comparatorClass = comparatorClass
        self.adderclass = adderclass
        super().__init__(**kwargs)

    def _compute(self, Xs, keys, deltas, rot, M, out_reg):
        if out_reg.num_qubits != 1:
            raise ValueError("The output must be a single qubit")

        neighbours = HoleCompute(name="neighbours", comparatorClass=self.comparatorClass, adderclass=self.adderclass)
        neighbours.compute(Xs, keys, deltas, rot, M)
        self.comparison = self.comparatorClass(name="compare_accum")
        self.comparison.compute(neighbours.accumulator, 1)        
        out_reg.x(cond=self.comparison.targ)
        out_reg.x()
        self.comparison.uncompute()
        neighbours.uncompute()
        self.set_result_qreg(out_reg)


## 5. Validate the actual circuit, then make a picture

A mathematical proof explains why the construction works; tests check whether our implementation matches that construction. In particular, an incorrect packing order or rotation direction could still give a permutation and pass the centre-count test. We therefore compare **every pixel** of a tiny circuit with the independently written classical reference, and test both initial output values.

### 5.1 Capture once, replay many times

Workbench's capture mechanism records instructions for reuse. `start_capture()` does not execute them by default; after `end_capture`, `playback_capture` executes the recorded program. Every query, including the first, must use playback.

The helper below allocates X before Y. Their union uses the corresponding low-to-high bit order, matching $a=x+2^{n_x}y$. These are register views on the same qubits, not an additional address register.

We return the context and registers explicitly so their lifetime is clear. The generously sized QPU capacity provides room for ancillary qubits; it is not a claim about the oracle's minimum width.


In [ ]:
def capture_hole_oracle(nx, ny, keys, M, deltas, rot=(1, 1), adderclass=CuccaroAdd):
    """Construct and capture one oracle; no input preparation is recorded."""
    if nx < 1 or ny < 1 or (nx + ny) % 2:
        raise ValueError("Use positive coordinate widths and an even total width")
    if any(len(delta) != 2 for delta in deltas):
        raise ValueError("This two-dimensional helper needs two-component offsets")
    if any(not 0 <= int(key) < 2**((nx + ny) // 2) for key in keys):
        raise ValueError("Each key must fit in one permutation half")
    qpu = QPU(num_qubits=8 * (nx + ny) + 2 * len(deltas) + 32, filters=[">>buffer>>", ">>bit-sim>>"])
    X = Qubits(nx, "X", qpu)
    Y = Qubits(ny, "Y", qpu)
    out = Qubits(1, "void", qpu)
    capture = qpu.start_capture()
    HoleOracle(name="hole_oracle", adderclass=adderclass).compute([X, Y], keys=tuple(keys), deltas=tuple(deltas), rot=rot, M=M, out_reg=out)
    qpu.end_capture(capture)
    return qpu, X, Y, out, capture


def query_captured(program, x, y, initial_output=0):
    """Basis-state test harness; preparation and reads stay outside the oracle."""
    qpu, X, Y, out, capture = program
    X.write(x)
    Y.write(y)
    out.write(initial_output)
    qpu.playback_capture(capture)
    if (X.read(), Y.read()) != (x, y):
        raise AssertionError(f"Input coordinate changed at {(x, y)}")
    return out.read()


### 5.2 Exhaustive tests on the running example

For each coordinate, the output must be `initial_output XOR reference_void`. We test both output values and apply the captured oracle twice: its XOR action implies $O_g^2=I$.

The tests explicitly check the output and coordinate restoration. Repeated queries also exercise reuse of the internal ancillary qubits, but these basis-state tests do not by themselves establish coherent phase behaviour or inspect every internal qubit.


In [ ]:
toy_program = capture_hole_oracle(toy_nx, toy_ny, toy_keys, toy_M, toy_deltas, toy_rot)
circuit_void = np.empty_like(toy_void)
for y in range(4):
    for x in range(4):
        for initial in (0, 1):
            actual = query_captured(toy_program, x, y, initial)
            assert actual == (initial ^ int(toy_void[y, x])), (x, y, initial, actual)
            toy_program[0].playback_capture(toy_program[4])
            assert toy_program[3].read() == initial
            assert (toy_program[1].read(), toy_program[2].read()) == (x, y)
        circuit_void[y, x] = query_captured(toy_program, x, y)
np.testing.assert_array_equal(circuit_void, toy_void)
print("Passed: 16 coordinates, both output values, coordinate restoration, and two-call identity.")


**Check the counting guarantee separately.** A stencil containing only $(0,0)$ makes the hole oracle equal the centre predicate. Sweep the threshold through empty, nonempty, and full cases. This also exercises the endpoint handling in `ShiftCentreCompute`.


In [ ]:
for centre_threshold in (0, 3, 7, 16):
    centre_program = capture_hole_oracle(2, 2, toy_keys, centre_threshold, ((0, 0),), toy_rot)
    observed = sum(query_captured(centre_program, x, y) for y in range(4) for x in range(4))
    assert observed == centre_threshold
    print(f"M = {centre_threshold:2d}: exactly {observed:2d} centres")


### 5.3 Make a small circuit-generated material

The default 8 × 8 grid keeps the scan small. Unlike the early preview, every pixel here comes from the actual reversible circuit. We display the classical centres alongside it, and assert agreement with the classical void map.

A complete scan makes $N$ oracle queries and is intended for verification and visualization. Later resource estimates concern **one query**, independent of whether we draw an image.


In [ ]:
nx, ny = 3, 3
keys = (6, 3, 5)
M = 8
deltas = ((0, 0), (-1, 0), (0, 1), (0, -1))
rot = (1, 1)
expected_centres, expected_void = geometry_reference(nx, ny, keys, M, deltas, rot)
small_program = capture_hole_oracle(nx, ny, keys, M, deltas, rot)
small_void = np.array([[query_captured(small_program, x, y) for x in range(2**nx)] for y in range(2**ny)], dtype=np.uint8)
np.testing.assert_array_equal(small_void, expected_void)
print("Classical reference and circuit outputs match.")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, array, title, cmap in zip(axes, (expected_centres, 1 - small_void), ("Classical centre predicate", "Material: black means void"), ("Blues", "gray")):
    ax.imshow(array, cmap=cmap, origin="lower", interpolation="nearest", vmin=0, vmax=1)
    ax.set(title=title, xlabel="x", ylabel="y")
plt.tight_layout()
plt.show()


**Optional larger scan.** Enable `RUN_FULL_CIRCUIT_IMAGE` to query the 64 × 64 preview through the circuit and verify every pixel. This requires 4096 queries and can take substantially longer than the small scan. The original key values fit six-bit halves; if you reduce widths, also choose keys that fit the new half width and adjust $M$.


In [ ]:
if RUN_FULL_CIRCUIT_IMAGE:
    full_program = capture_hole_oracle(preview_nx, preview_ny, preview_keys, preview_M, preview_deltas, preview_rot)
    full_void = np.array([[query_captured(full_program, x, y) for x in range(2**preview_nx)] for y in range(2**preview_ny)], dtype=np.uint8)
    np.testing.assert_array_equal(full_void, preview_void)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(1 - full_void, cmap="gray", origin="lower", interpolation="nearest", vmin=0, vmax=1)
    # ax.set(title="64 × 64 material from circuit queries", xlabel="x", ylabel="y")
    fig.tight_layout()
    ax.axis('off')
    # black outline around the image
    h, w = full_void.shape[:2]
    import matplotlib.patches as patches
    rect = patches.Rectangle(
        (-0.5, -0.5), w, h,
        linewidth=2, edgecolor='black', facecolor='none'
    )
    ax.add_patch(rect)
    # fig.show()
    if SAVE_PAPER_FIGURES:
        namefile = "full_cheese.pdf"
        fig.savefig(export_dir / fig_dir / namefile)
    plt.show()


## 6. Use Construct to understand the circuit's cost

We now change from testing correctness to estimating resources. Build a **fresh** program containing just one oracle call, with no input writes, output reads, or repeated test queries. Otherwise resource accounting can include the testing workload rather than the query we want to study.

For numeric estimates, no simulator is necessary. We retain a buffer so this same one-query program can also be drawn and exported. The witness filter supplies resource accounting without evaluating a basis state.

You can export the full circuit as a file and inspect it in the [Circuit Designer](https://construct.psiquantum.com/cd).

In [ ]:
def build_estimate(nx, ny, keys, M, deltas, rot=(1, 1), adderclass=CuccaroAdd):
    """Record one complete oracle query, with resource accounting and no simulator."""
    qpu = QPU(num_qubits=3 * (nx + ny) + 2 * len(deltas) + 32, filters=[">>witness>>", ">>buffer>>"])
    X = Qubits(nx, "X", qpu)
    Y = Qubits(ny, "Y", qpu)
    out = Qubits(1, "void", qpu)
    HoleOracle(name="hole_oracle", adderclass=adderclass).compute([X, Y], keys=tuple(keys), deltas=tuple(deltas), rot=rot, M=M, out_reg=out)
    return qpu, resource_estimator(qpu).resources()


cost_qpu, one_query_resources = build_estimate(3, 3, (6, 3, 5), 8, deltas)
display(pd.Series(one_query_resources, name="One complete oracle query"))

### 6.1 Read the metrics before comparing them

| Metric | Interpretation |
| --- | --- |
| `qubit_highwater` | Peak allocated logical qubits, including inputs, output, and ancillary qubits |
| `active_volume` | Computational work measured in logical blocks in the active-volume architecture |
| `toffs` | Toffoli gates: controlled-controlled X operations |
| `gidney_lelbows`, `gidney_relbows` | Temporary-AND computation and erasure primitives used by some arithmetic implementations |

A logical qubit is the error-corrected qubit used by the algorithm. Active volume sums the logical blocks associated with operations, excluding idle storage; it is not a time in seconds. See [Litinski and Nickerson, *Active volume*](https://arxiv.org/abs/2211.15465).

QPU capacity is the allocation limit supplied to the program, not the measured peak. Highwater includes the coordinate and output registers. To count only ancillary qubits for an isolated two-register $q$-bit adder, subtract its $2q$ input qubits. For a geometry oracle, subtract its $n$ input qubits and one output qubit. Arithmetic ancillary qubits can be reused, so equal whole-oracle peaks do not imply equal ancillary requirements for each adder.

Explicit decompositions can change the reported allocation and gate categories. A Toffoli count alone can miss costs represented as Gidney elbows. We use active volume and peak qubits for the main plots, as in the paper.

[Resource-estimation definitions and decomposition](https://docs.construct.psiquantum.com/psiqdk-workbench/new-tutorials/Resource-Estimates-Basic-Numerics.html).


### 6.2 Explore the hierarchy with Resource Analyzer

Enable the viewer or exports at the top of the notebook. Exporting creates a `.qre-analysis` file that you can open in [Resource Analyzer](https://construct.psiquantum.com/ra). The in-notebook `CallGraph` displays the same analysis file.

**Try this in the viewer:**

1. Find `hole_oracle`, then expand `neighbours`, the count computation.
2. Identify each `offset_*` query, its controlled accumulator increment, and its uncomputation.
3. Expand a permutation and identify its adders.
4. Switch between active volume and the reported gate categories.
5. Check whether the blocks you expected to dominate actually dominate.

Keep the hierarchy in mind: improving a frequently repeated subroutine can matter more than improving a rare operation.


In [ ]:
if RUN_INTERACTIVE_VIEWERS or EXPORT_CONSTRUCT_FILES:
    from psiqdk.workbench.integrations import resource_analyzer
    export_dir = Path("quantum_cheese_exports")
    export_dir.mkdir(exist_ok=True)
    resource_analyzer.export(cost_qpu, str(export_dir / "hole_oracle"))
    print("Exported", export_dir / "hole_oracle.qre-analysis")
    if RUN_INTERACTIVE_VIEWERS:
        from psiqdk.visualize import CallGraph
        display(CallGraph(src=str(export_dir / "hole_oracle.qre-analysis")))


### 6.3 Compare interchangeable arithmetic routines

`CuccaroAdd` is one implementation of modular addition. The routines below implement the same register update with different internal circuits. Compare them at identical widths and through the same estimation pipeline.

The benchmark has two $q$-qubit input registers and no separate output register. Its highwater therefore includes $2q$ input qubits; the remaining allocation is ancillary. An isolated adder's ancillary requirement need not determine the peak of the complete geometry oracle.

Expect a trade-off rather than a universal winner. Compare active volume and peak qubits, and consult Toffoli and elbow counts when explaining the difference.


In [ ]:
adder_rows = []
for adderclass in (CuccaroAdd, GidneyAdd, ConditionallyCleanAdd, NaiveAdd):
    for width in (3, 5, 7):
        adder_qpu = QPU(num_qubits=8 * width + 16, filters=[">>witness>>", ">>buffer>>"])
        lhs = Qubits(width, "lhs", adder_qpu)
        rhs = Qubits(width, "rhs", adder_qpu)
        adderclass(name="modular_add").compute(lhs, rhs)
        resources = resource_estimator(adder_qpu).resources()
        adder_rows.append({"width": width, "adder": adderclass.__name__, **resources, "ancillary_qubits": resources["qubit_highwater"] - 2 * width})
adder_df = pd.DataFrame(adder_rows)
display(adder_df)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, metric, label in zip(axes, ("active_volume", "qubit_highwater"), ("Active volume", "Peak logical qubits")):
    for name, group in adder_df.groupby("adder"):
        ax.plot(group["width"], group[metric], marker="o", label=name)
    ax.set(xlabel="Bits per register", ylabel=label)
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()


### 6.4 Derive the scaling before plotting

Let $G_Q(n,m)$ count the gates in a forward shifted centre query, and let $G_a(S)$ bound a controlled addition on the accumulator. Computing the count over $R$ offsets requires
$$
G_Z\le R\bigl(2G_Q(n,m)+G_a(S)\bigr).
$$
Each contribution includes the query, the controlled increment, and the inverse query. The complete oracle computes the count, compares it, and reverses the count computation:
$$
G_G\le2G_Z+G_c(S)+O(1),
$$
where $G_c(S)$ bounds the final comparison and its uncomputation. These are gate-count bounds; measurement-assisted inverse routines and active-volume estimates can have different costs.

For linear-size arithmetic on a fixed number of coordinate axes,
$$
G_Q=O(mn+n),\qquad G_G=O\!\left(R(mn+S)\right).
$$
Here $n=\log_2N$. For a fixed stencil, $R$ and $S$ are constant as the material grows. Choosing $m=\Theta(n)$ still gives polynomial-size circuits. The paper also describes arithmetic choices with polylogarithmic depth; those are not the routines benchmarked here.

The count uses $S=\lfloor\log_2R\rfloor+2$ qubits for $R\ge1$, matching `R.bit_length() + 1`. Each centre flag is used and uncomputed before the next offset, so there is no $R$-qubit flag register. The ancillary allocation includes the accumulator plus reusable comparison and adder registers. Peak total qubits also include the $n$ coordinate qubits and the output qubit.


### 6.5 Optional controlled experiments

Enable `RUN_RESOURCE_SWEEPS` to produce three independent sweeps for both `CuccaroAdd` and `GidneyAdd`, using identical inputs for each adder. The table labels the adder and each plot overlays both implementations. Each sweep uses a centre fraction of $1/8$, avoiding an unintended change in density as the grid grows.

| Sweep | Vary | Hold fixed | Prediction to investigate |
| --- | --- | --- | --- |
| Address size | Bits per axis | Three rounds, two offsets | Arithmetic cost grows roughly linearly with total address width |
| Mixing | Number of rounds | Grid and stencil | Additional rounds add arithmetic and may improve spatial mixing |
| Shape size | Number of offsets | Grid and rounds | More offsets add queries; the accumulator width grows in steps as larger counts must fit |

Small constant effects and the bit patterns of keys and thresholds can affect the curves. Inspect these as finite circuit estimates, rather than fitting an asymptotic theorem to a few data points.


In [ ]:
if RUN_RESOURCE_SWEEPS:
    sweep_rows = []
    for adderclass in (CuccaroAdd, GidneyAdd):
        for width in (2, 3, 4, 5, 6):
            schedule = tuple(int(k) for k in np.random.default_rng(42).integers(0, 2**width, size=3))
            _, resources = build_estimate(width, width, schedule, 2**(2 * width - 3), ((0, 0), (1, 0)), adderclass=adderclass)
            sweep_rows.append({"sweep": "Bits per axis", "value": width, "adder": adderclass.__name__, **resources})
        width = 4
        schedule = tuple(int(k) for k in np.random.default_rng(42).integers(0, 16, size=6))
        for rounds in (1, 2, 3, 4, 6):
            _, resources = build_estimate(width, width, schedule[:rounds], 32, ((0, 0), (1, 0)), adderclass=adderclass)
            sweep_rows.append({"sweep": "Rounds", "value": rounds, "adder": adderclass.__name__, **resources})
        for count in (1, 2, 3, 4, 6):
            stencil = tuple((offset, 0) for offset in range(count))
            _, resources = build_estimate(width, width, schedule[:3], 32, stencil, adderclass=adderclass)
            sweep_rows.append({"sweep": "Stencil offsets", "value": count, "adder": adderclass.__name__, **resources})
    sweep_df = pd.DataFrame(sweep_rows)
    # display(sweep_df)
    fig, axes = plt.subplots(2, 3, figsize=(9, 6),sharey="row",sharex="col")
    for col_id, sweep in enumerate(("Bits per axis", "Rounds", "Stencil offsets")):
        group = sweep_df[sweep_df["sweep"] == sweep]
        for ax, metric in zip(axes[:,col_id], ("active_volume", "qubit_highwater")):
            for adder, adder_group in group.groupby("adder", sort=False):
                ax.plot(adder_group["value"], adder_group[metric], marker="o", label=adder)
            # ax.set(xlabel=sweep, ylabel=metric)
            ax.legend()
            ax.grid(alpha=0.25)
            if col_id == 0:
                ax.set(ylabel=metric)
        ax.set(xlabel=sweep)
        
    fig.tight_layout()
    if SAVE_PAPER_FIGURES:
        namefile = "sweep_plot.pdf"
        fig.savefig(export_dir / fig_dir / namefile)
    fig.show()


## 7. Experiment with your material

At this point you can explain the geometry, implement its coherent query, check it, and inspect its cost. The following experiments use the fast classical reference to explore the model. For a candidate you want to compile, pass the same parameters to `capture_hole_oracle` and `build_estimate`.

### 7.1 Centres are nested when only M changes

Fix the permutation and stencil. If $M_1\le M_2$, then $c_{M_1}(x)\le c_{M_2}(x)$ at every site. The unit-weight counts and their threshold at one preserve this ordering, so the void sets are also nested. This monotonicity need not hold for signed kernels in Section 8. The void fraction is **nondecreasing**, though overlaps can create plateaus.

Changing keys changes the realization instead, so that pointwise ordering no longer follows. Exact centre count remains guaranteed for every key choice.


In [ ]:
thresholds = (128, 256, 512, 768)
fig, axes = plt.subplots(1, len(thresholds), figsize=(12, 3.2))
previous_void = None
for ax, threshold in zip(axes, thresholds):
    centres, void = geometry_reference(6, 6, preview_keys, threshold, preview_deltas, preview_rot)
    assert centres.sum() == threshold
    if previous_void is not None:
        assert np.all(previous_void <= void)
    previous_void = void
    ax.imshow(1 - void, cmap="gray", origin="lower", interpolation="nearest", vmin=0, vmax=1)
    ax.set(title=f"M={threshold}; void={void.mean():.1%}", xlabel="x", ylabel="y")
plt.tight_layout()
plt.show()


### 7.2 Change the stencil, not the centre process

A horizontal pair, a cross, and a discrete disk produce different holes from the same centres. The disk below means all integer offsets whose squared distance is at most $2^2$.

**Try it:** predict which stencil has the largest circuit cost before building it. For this implementation, the number of offsets directly controls the number of shifted centre queries. Hole overlap affects the picture but does not skip circuit branches at execution time.


In [ ]:
stencils = {"Horizontal pair": ((0, 0), (1, 0)), "Cross": ((0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)), "Disk, radius 2": tuple((dx, dy) for dy in range(-2, 3) for dx in range(-2, 3) if dx * dx + dy * dy <= 4)}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, (label, stencil) in zip(axes, stencils.items()):
    _, void = geometry_reference(6, 6, preview_keys, 256, stencil, preview_rot)
    ax.imshow(1 - void, cmap="gray", origin="lower", interpolation="nearest", vmin=0, vmax=1)
    ax.set(title=f"{label}: R={len(stencil)}", xlabel="x", ylabel="y")
plt.tight_layout()
plt.show()


### 7.3 A short set of investigations

- **Keys:** change the key sequence while holding everything else fixed. Verify centre count and compare void fraction.
- **Rounds:** use one, two, and five rounds. Look for visible structure, then compare resource estimates. A better-looking picture is not a proof of pseudorandomness.
- **Periodic wrapping:** use a long horizontal stencil and identify holes continuing across the left/right boundary.
- **Three dimensions:** supply three coordinate registers and three-component offsets to the ND routines. Keep the total address width even. The two-dimensional packing and plotting helpers need replacement.
- **Arithmetic choice:** replace the adder in a complete oracle, check correctness on a tiny grid again, and compare the whole-query cost rather than only an isolated adder.


## 8. Going further: weighted local rules

The circuit already accumulates contributions. Fixed-shape holes use weight 1 for every offset and threshold 1. To shape more general textures, assign an integer weight $\alpha_r$ to offset $\eta_r$:
$$
s(x)=\sum_{r=1}^R\alpha_r c_{k,M}(x-\eta_r),\qquad
g_{\Lambda,\tau}(x)=\mathbf1[s(x)\ge\tau].
$$
The list $\Lambda=\{(\eta_r,\alpha_r)\}_{r=1}^R$ is a **kernel**. Its weighted neighbourhood sum is a discrete **convolution**, with periodic wrapping. The threshold $\tau$ decides which scores are void. The paper's locally textured family keeps the kernel and threshold fixed as the domain grows.

Positive weights reinforce a contribution to the void score; negative weights oppose it. The example below uses a positive centre weight, positive weights on its four axial neighbours, and negative weights on its four diagonal neighbours. We threshold at $\tau=0$. This is a texture model, not a simulation of material formation dynamics.


In [ ]:
def score_reference(centres, weighted_offsets):
    """Evaluate an integer local convolution on a periodic grid."""
    score = np.zeros_like(centres, dtype=np.int64)
    for (dx, dy), weight in weighted_offsets:
        score += int(weight) * np.roll(centres, shift=(dy, dx), axis=(0, 1)).astype(np.int64)
    return score


unit_kernel = tuple((delta, 1) for delta in preview_deltas)
weighted_centres, unit_void = geometry_reference(preview_nx, preview_ny, preview_keys, 2**(preview_nx + preview_ny) // 2, preview_deltas, preview_rot)
assert np.array_equal(score_reference(weighted_centres, unit_kernel) >= 1, unit_void)
# weighted_kernel = (((0, 0), 2), ((1, 0), 1), ((-1, 0), 1), ((0, 1), 1), ((0, -1), 1), ((2, 0), -2), ((-2, 0), -2))
weighted_kernel = (((0, 0), 4),((1, 0), 1), ((-1, 0), 1), ((0, 1), 1), ((0, -1), 1), ((1, 1), -2), ((-1, 1), -2), ((-1, -1), -2), ((1, -1), -2))
kernel_image = np.zeros((3, 3))
for xy,w in weighted_kernel:
    kernel_image[xy[1]+1, xy[0]+1] = w
score = score_reference(weighted_centres, weighted_kernel)
weighted_tau = 0
weighted_void = score >= weighted_tau
fig, axes = plt.subplots(1, 4, figsize=(11, 3.5))
axes[0].imshow(weighted_centres, cmap="Blues", origin="lower", vmin=0, vmax=1)
img1 = axes[1].imshow(kernel_image, cmap="coolwarm", origin="lower",vmin=-4, vmax=4)
fig.colorbar(img1, ax=axes[1], shrink=0.75)
img2 = axes[2].imshow(score, cmap="coolwarm", origin="lower",vmin=-8, vmax=8)
fig.colorbar(img2, ax=axes[2], shrink=0.75)
axes[3].imshow(1 - weighted_void, cmap="gray", origin="lower", vmin=0, vmax=1)
for ax, title in zip(axes, ("Centres", "Weighted kernel", "Signed local score", f"Void where score ≥ {weighted_tau}")):
    ax.set(title=title, xlabel="x", ylabel="y")
plt.tight_layout()
plt.show()


### Circuit extension: a guided exercise

Generalize the accumulator you have already built:

1. Replace each controlled increment by a controlled addition of its integer weight $\alpha_r$.
2. Choose $S$ so that $2^{S-1}>W_\Lambda$, where $W_\Lambda=\sum_r|\alpha_r|$. The code expression `max(2, W.bit_length() + 1)` suffices when `W` is that sum. Every partial score then fits in the signed register.
3. Compare the signed score with $\tau$, XOR the answer into the output, uncompute the comparison, then reverse the accumulation.

The adder operates modulo $2^S$. Represent a negative weight by `weight % (1 << S)`. Two's-complement encoding then gives the intended signed sum without overflow.

An unsigned comparison of a negative score's bit pattern is wrong. For a threshold in the signed range, flip the accumulator's most significant bit and compare the resulting unsigned value with $\tau+2^{S-1}$. Complement the less-than result to obtain $[s\ge\tau]$. Uncompute the comparator before restoring that bit, then reverse the accumulation. If $\tau>W_\Lambda$, the result is always 0; if $\tau\le-W_\Lambda$, it is always 1.

**Acceptance checks:** unit weights and threshold one must reproduce `HoleOracle`. A tiny signed kernel must match `score_reference` at every pixel for both initial output values, with coordinates and ancillary qubits restored. Include all uncomputation in the resource estimate.

The weighted classical example is executable. The signed circuit is left as an exercise following the same construction as the paper.


## 9. What you can now explain and reuse

You have built a compact geometry description from a keyed permutation, a centre threshold, and a local shape. The count of centres follows from bijectivity, while the shape is the union of translated stencils. The circuit counts covering centres, thresholds at one, and reverses the count computation. The reversible implementation evaluates that same definition and cleans its internal work registers.

The Construct workflow you used is reusable: **write a Qubrick, simulate small instances, inspect the hierarchy, estimate a clean one-query program, and compare implementations under the same assumptions**. A material picture checks outputs; Circuit Designer explains circuit structure; Resource Analyzer helps identify expensive components.


### Further reading

- [*How to make quantum cheese*](https://arxiv.org/abs/2606.00222): the source construction and its broader algorithmic setting.
- [Construct overview](https://docs.construct.psiquantum.com/): the software suite and its components.
- [PsiQDK installation](https://docs.construct.psiquantum.com/psiqdk/installation.html).
- [Workbench execution pipelines](https://docs.construct.psiquantum.com/psiqdk-workbench/new-tutorials/Configuring-Execution.html).
- [Qubrick uncomputation](https://docs.construct.psiquantum.com/psiqdk-workbench/new-tutorials/Qubricks-Uncomputation.html).
- [Numeric resource estimates](https://docs.construct.psiquantum.com/psiqdk-workbench/new-tutorials/Resource-Estimates-Basic-Numerics.html).
- [Circuit Designer](https://construct.psiquantum.com/cd) and [Resource Analyzer](https://construct.psiquantum.com/ra): open exported programs and analyses.
